# Guitar MIDI — pipeline Kaggle

Ce notebook monte un snapshot source privé attaché et lance soit le smoke test, soit le train complet, soit la reconstruction de `data/processed`. Il n'utilise aucun accès réseau au runtime. Le package de train est refusé s'il contient le split test.

In [ ]:
TASK = "smoke"  # "smoke", "train", "rank", "select" ou "rebuild"
BRANCH = "codex/dual-stream-bass"
SOURCE_DATASET_SLUG = ""  # injecté par le publisher Kaggle
CONFIG_PATH = "configs/polyphonic_train.yaml"
INITIAL_CHECKPOINT_NAME = ""
WORKSPACE = "/kaggle/working/midi"
WORKERS = 4
SMOKE_EXAMPLES = 8192
SMOKE_VALIDATION_EXAMPLES = 2048
LOG_EVERY_BATCHES = 25
MAXIMUM_RUNTIME_MINUTES = 30.0
MAXIMUM_EXAMPLES = 60_000
MAXIMUM_RECORDINGS = 12
MAXIMUM_CANDIDATES = 8


In [ ]:
import importlib
import json
import os
import pathlib
import shutil
import subprocess
import sys
import tarfile

os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True, write_through=True)
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(line_buffering=True, write_through=True)
if not ((3, 9) <= sys.version_info[:2] < (3, 13)):
    raise RuntimeError(f"Python incompatible: {sys.version.split()[0]}")
workspace = pathlib.Path(WORKSPACE)
if workspace.exists():
    shutil.rmtree(workspace)
input_root = pathlib.Path("/kaggle/input")
if not SOURCE_DATASET_SLUG:
    raise RuntimeError("SOURCE_DATASET_SLUG was not injected by the Kaggle publisher")
expected_source_root = input_root / SOURCE_DATASET_SLUG
mounted_roots = sorted(path for path in input_root.iterdir() if path.is_dir())
source_archives = sorted(input_root.rglob("midi_source.tar.gz"))
source_trees = sorted(path for path in input_root.rglob("midi_source") if path.is_dir())
expanded_source = sorted(set(
    marker.parent for marker in input_root.rglob("pyproject.toml")
    if (marker.parent / "src").is_dir()
    and (marker.parent / "scripts/cloud/kaggle_entrypoint.py").is_file()
))
source_candidates = sorted(set(source_archives + source_trees + expanded_source))
print(json.dumps({
    "expected_source_root": str(expected_source_root),
    "mounted_inputs": [path.name for path in mounted_roots],
    "source_candidates": [str(path) for path in source_candidates],
}, indent=2))
if len(source_candidates) != 1:
    raise RuntimeError(f"Expected one offline source snapshot, got {source_candidates}")
source_snapshot = source_candidates[0]
metadata_root = source_snapshot if source_snapshot.is_dir() else source_snapshot.parent
metadata_candidates = sorted(metadata_root.rglob("source_metadata.json"))
if len(metadata_candidates) > 1:
    raise RuntimeError(f"Expected at most one source metadata file, got {metadata_candidates}")
if metadata_candidates:
    source_metadata = json.loads(metadata_candidates[0].read_text(encoding="utf-8"))
else:
    source_metadata = {
        "branch": BRANCH,
        "commit": SOURCE_DATASET_SLUG.rsplit("-", 1)[-1],
    }
if source_snapshot.is_dir():
    shutil.copytree(source_snapshot, workspace)
else:
    workspace.mkdir(parents=True)
    with tarfile.open(source_snapshot, "r:gz") as archive:
        destination = workspace.resolve()
        for member in archive.getmembers():
            target = (workspace / member.name).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Source archive escapes workspace: {member.name}")
        archive.extractall(workspace)
os.environ["GUITAR_MIDI_SOURCE_BRANCH"] = source_metadata["branch"]
os.environ["GUITAR_MIDI_SOURCE_COMMIT"] = source_metadata["commit"]
required_imports = ["numpy", "scipy", "soundfile", "pandas", "librosa", "matplotlib", "tqdm", "yaml", "tensorflow", "keras"]
versions = {}
for module_name in required_imports:
    module = importlib.import_module(module_name)
    versions[module_name] = getattr(module, "__version__", "available")
print(json.dumps({"runtime": sys.version, "packages": versions}, indent=2), flush=True)
gpu_probe = subprocess.run(["nvidia-smi"], check=False, capture_output=True, text=True)
print("NVIDIA_SMI_BEGIN", flush=True)
print(gpu_probe.stdout or gpu_probe.stderr, flush=True)
print(f"NVIDIA_SMI_END returncode={gpu_probe.returncode}", flush=True)


In [ ]:
command = [
    sys.executable,
    str(workspace / "scripts/cloud/kaggle_entrypoint.py"),
    "--task", TASK,
    "--config", CONFIG_PATH,
    "--input-root", "/kaggle/input",
    "--workers", str(WORKERS),
    "--smoke-examples", str(SMOKE_EXAMPLES),
    "--smoke-validation-examples", str(SMOKE_VALIDATION_EXAMPLES),
    "--log-every-batches", str(LOG_EVERY_BATCHES),
    "--maximum-runtime-minutes", str(MAXIMUM_RUNTIME_MINUTES),
    "--maximum-examples", str(MAXIMUM_EXAMPLES),
    "--maximum-recordings", str(MAXIMUM_RECORDINGS),
    "--maximum-candidates", str(MAXIMUM_CANDIDATES),
]
if INITIAL_CHECKPOINT_NAME:
    command.extend(["--initial-checkpoint-name", INITIAL_CHECKPOINT_NAME])
subprocess.run(command, cwd=workspace, check=True)
subprocess.run([
    sys.executable,
    str(workspace / "scripts/cloud/package_kaggle_outputs.py"),
    "--task", TASK,
    "--output-dir", "/kaggle/working/guitar-midi-results",
], cwd=workspace, check=True)


In [ ]:
results = pathlib.Path("/kaggle/working/guitar-midi-results")
materialized_input = pathlib.Path("/kaggle/working/guitar-midi-input")
if materialized_input.exists():
    shutil.rmtree(materialized_input)
if workspace.exists():
    shutil.rmtree(workspace)
for path in sorted(results.iterdir()):
    print(f"{path.name}: {path.stat().st_size} bytes")
